# Writing a morphology-with-spines file

This notebook demonstrates how to build a morph-with-spines HDF5 file from
scratch using the writer API, then validate the result.

In [ ]:
import numpy as np
import pandas as pd

from morph_spines import (
    validate_morph_with_spines_file,
    write_morphology,
    write_soma_mesh,
    write_spine_meshes,
    write_spine_skeletons,
    write_spine_table,
)

## Define the output file and neuron name

In [ ]:
output_file = "./my_morph_with_spines.h5"
neuron_name = "my_neuron"

## Write the neuron morphology

The morphology skeleton follows the H5v1 format: a `points` array (N x 4:
x, y, z, radius) and a `structure` array (M x 3: offset, type, parent).

In [ ]:
# A simple Y-shaped morphology: 3-point soma + 2 dendrite sections
points = np.array(
    [
        [0.0, 0.0, 0.0, 5.0],  # soma point 0
        [0.0, 5.0, 0.0, 5.0],  # soma point 1
        [0.0, -5.0, 0.0, 5.0],  # soma point 2
        [0.0, 5.0, 0.0, 1.0],  # section 0 start
        [10.0, 10.0, 0.0, 0.8],  # section 0 end
        [10.0, 10.0, 0.0, 0.8],  # section 1 start
        [20.0, 15.0, 0.0, 0.5],  # section 1 end
        [10.0, 10.0, 0.0, 0.8],  # section 2 start
        [20.0, 5.0, 0.0, 0.5],  # section 2 end
    ],
    dtype=np.float32,
)

# Structure: [point_offset, section_type, parent_section]
structure = np.array(
    [
        [0, 1, -1],  # soma (3 points)
        [3, 3, 0],  # section 0: basal dendrite, parent=soma
        [5, 3, 1],  # section 1: parent=section 0
        [7, 3, 1],  # section 2: parent=section 0
    ],
    dtype=np.int32,
)

write_morphology(output_file, neuron_name, points, structure)
print("Neuron morphology written.")

## Write the spines table

The spines table is a DataFrame with one row per spine. All mandatory columns
must be present. The `spine_morphology` column references the skeleton/mesh
group name, and `spine_id` is the 0-based index within that group.

In [ ]:
n_spines = 3

spines_table = pd.DataFrame(
    {
        "afferent_surface_x": [12.0, 14.0, 16.0],
        "afferent_surface_y": [11.5, 12.5, 8.0],
        "afferent_surface_z": [0.5, 0.3, -0.2],
        "afferent_center_x": [12.0, 14.0, 16.0],
        "afferent_center_y": [11.0, 12.0, 7.5],
        "afferent_center_z": [0.0, 0.0, 0.0],
        "spine_morphology": [neuron_name] * n_spines,
        "spine_id": np.arange(n_spines, dtype=np.uint32),
        "spine_length": [1.2, 1.5, 0.9],
        "spine_orientation_vector_x": [0.0, 0.0, 0.0],
        "spine_orientation_vector_y": [1.0, 1.0, -1.0],
        "spine_orientation_vector_z": [0.0, 0.0, 0.0],
        "spine_rotation_x": [0.0, 0.0, 0.0],
        "spine_rotation_y": [0.0, 0.0, 0.0],
        "spine_rotation_z": [0.0, 0.0, 0.0],
        "spine_rotation_w": [1.0, 1.0, 1.0],
        "afferent_section_id": np.array([2, 2, 3], dtype=np.uint32),
        "afferent_segment_id": np.array([0, 0, 0], dtype=np.int32),
        "afferent_segment_offset": [0.3, 0.7, 0.5],
        "afferent_section_pos": [0.3, 0.7, 0.5],
    }
)

write_spine_table(output_file, neuron_name, spines_table)
print(f"Spines table written ({n_spines} spines).")

## Write spine skeletons

Spine skeletons use the same H5v1 format. Each root section in the structure
corresponds to one spine (indexed by `spine_id`).

In [ ]:
# 3 spines, each with 2 points (base and tip)
skel_points = np.array(
    [
        [0.0, 0.0, 0.0, 0.2],
        [0.0, 1.2, 0.0, 0.1],  # spine 0
        [0.0, 0.0, 0.0, 0.2],
        [0.0, 1.5, 0.0, 0.1],  # spine 1
        [0.0, 0.0, 0.0, 0.2],
        [0.0, 0.9, 0.0, 0.1],  # spine 2
    ],
    dtype=np.float32,
)

# Each spine is a root section (parent = -1)
skel_structure = np.array(
    [
        [0, 2, -1],  # spine 0: starts at point 0
        [2, 2, -1],  # spine 1: starts at point 2
        [4, 2, -1],  # spine 2: starts at point 4
    ],
    dtype=np.int32,
)

write_spine_skeletons(output_file, neuron_name, skel_points, skel_structure)
print("Spine skeletons written.")

## Write spine meshes

Spine meshes are stored as concatenated vertices/triangles with an offsets
array. Triangle indices are local to each spine's vertex slice.

Optionally, head/neck triangle classification can be included via a third
column in the offsets array and a `head_neck_values` dataset. Set the flag
below to control which variant is written.

In [ ]:
# Set to True to include head/neck triangle classification
WRITE_HEAD_NECK = True

The following cell creates the data without head/neck triangle classification. Writing to disk happens later, depending on the `WRITE_HEAD_NECK` flag.

In [ ]:
# Simple triangular prism for each spine (6 vertices, 8 triangles)
single_verts = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.1, 0.0, 0.0],
        [0.05, 0.0, 0.1],
        [0.0, 1.0, 0.0],
        [0.1, 1.0, 0.0],
        [0.05, 1.0, 0.1],
    ],
    dtype=np.float32,
)

single_tris = np.array(
    [
        [0, 1, 2],
        [3, 4, 5],  # caps
        [0, 1, 4],
        [0, 4, 3],  # side 1
        [1, 2, 5],
        [1, 5, 4],  # side 2
        [2, 0, 3],
        [2, 3, 5],  # side 3
    ],
    dtype=np.int32,
)

# Concatenate for all spines
all_vertices = np.tile(single_verts, (n_spines, 1))
all_triangles = np.tile(single_tris, (n_spines, 1))

# Offsets: [vertex_offset, triangle_offset] per spine + final entry
n_v = len(single_verts)
n_t = len(single_tris)
offsets = np.array(
    [[i * n_v, i * n_t] for i in range(n_spines + 1)],
    dtype=np.int32,
)

### Head/neck classification

When adding head/neck classification, the offsets array has a third column indexing
into a `head_neck_values` array. Within each spine's triangle range, triangles
must be sorted: neck first, then head(s).

The `head_neck_values` per spine is an offset-style array:
- `hn[0]`: end of undefined / start of neck
- `hn[1]`: end of neck / start of head
- `hn[-1]`: total triangle count for that spine

We show 3 cases: spine 0 has neck+head, spine 1 is neck-only, spine 2 is
head-only.

The following cell creates the data with head/neck triangle classification. Writing to disk happens later, depending on the `WRITE_HEAD_NECK` flag.

In [ ]:
# Spine 0: neck+head -> hn = [0, 4, 8] (4 neck, 4 head)
# Spine 1: neck-only -> hn = [0, 8, 8] (8 neck, 0 head)
# Spine 2: head-only -> hn = [0, 0, 8] (0 neck, 8 head)
hn_spine_0 = np.array([0, 4, 8], dtype=np.int32)
hn_spine_1 = np.array([0, 8, 8], dtype=np.int32)
hn_spine_2 = np.array([0, 0, 8], dtype=np.int32)

all_hn_values = np.concatenate([hn_spine_0, hn_spine_1, hn_spine_2])

# Offsets: 3 columns [vertex_offset, triangle_offset, hn_offset]
n_hn_per_spine = 3
offsets_hn = np.array(
    [
        [0 * n_v, 0 * n_t, 0 * n_hn_per_spine],
        [1 * n_v, 1 * n_t, 1 * n_hn_per_spine],
        [2 * n_v, 2 * n_t, 2 * n_hn_per_spine],
        [3 * n_v, 3 * n_t, 3 * n_hn_per_spine],
    ],
    dtype=np.int32,
)

Write spine meshes to disk, with or without head/neck classification, depending on `WRITE_HEAD_NECK` flag.

In [ ]:
if WRITE_HEAD_NECK:
    write_spine_meshes(
        output_file,
        neuron_name,
        all_vertices,
        all_triangles,
        offsets_hn,
        head_neck_values=all_hn_values,
    )
    print("Spine meshes written WITH head/neck classification:")
    print("  Spine 0: neck+head (4 neck tris, 4 head tris)")
    print("  Spine 1: neck-only (8 neck tris)")
    print("  Spine 2: head-only (8 head tris)")
else:
    write_spine_meshes(
        output_file,
        neuron_name,
        all_vertices,
        all_triangles,
        offsets,
    )
    print(f"Spine meshes written ({n_spines} spines, {len(all_vertices)} total vertices).")

## Write soma mesh (optional)

The soma mesh is a simple triangle mesh with global vertex indices.

In [ ]:
# Tetrahedron as a simple soma mesh
soma_vertices = np.array(
    [
        [0.0, 0.0, 5.0],
        [5.0, 0.0, -2.5],
        [-5.0, 0.0, -2.5],
        [0.0, 5.0, 0.0],
    ],
    dtype=np.float32,
)

soma_triangles = np.array(
    [
        [0, 1, 2],
        [0, 1, 3],
        [1, 2, 3],
        [2, 0, 3],
    ],
    dtype=np.int32,
)

write_soma_mesh(output_file, neuron_name, soma_vertices, soma_triangles)
print("Soma mesh written.")

## Validate the result

Run the file validator to confirm everything is correct.

In [ ]:
result = validate_morph_with_spines_file(output_file, check_data_integrity=True)
print(result)

## Read back the file

Verify the round-trip by loading the file we just wrote.

In [ ]:
from morph_spines import load_morphology_with_spines

m = load_morphology_with_spines(output_file, load_meshes=True)

print(f"Loaded {m.spines.spine_count} spines")
print(f"Morphology sections: {len(m.morphology.sections)}")
print(f"Soma mesh faces: {len(m.soma.soma_mesh.faces)}")
print(f"Spine 0 mesh faces: {len(m.spines.spine_mesh(0).faces)}")